# 02 — Fine-Tuning with Unsloth (Colab)

**Goal**: Fine-tune Llama 3.2 (3B) on a custom dataset using Google Colab's free T4 GPU.

**IMPORTANT**: This notebook is designed for **Google Colab**. 
Go to: https://colab.research.google.com → Upload → Run with T4 GPU.

In [ ]:
# @title 1. Install Unsloth (run once)
!pip install unsloth
# Also install xformers and bitsandbytes for QLoRA
!pip install xformers bitsandbytes

In [ ]:
# @title 2. Load Model with QLoRA
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Llama-3.2-3B-Instruct"  # Free, small, runs on T4
max_seq_length = 2048  # Context window
dtype = None  # Auto-detect
load_in_4bit = True  # QLoRA: 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded: {model_name}")
print(f"Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# @title 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # Scaling
    lora_dropout=0,  # 0 is optimized
    bias="none",
    use_gradient_checkpointing=True,  # Save memory
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA adapters added!")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable / 1e6:.2f}M / {total / 1e6:.2f}M ({100 * trainable / total:.2f}%)")

In [ ]:
# @title 4. Prepare Dataset
# We'll create a simple Q&A dataset about GenAI concepts
from datasets import Dataset
import pandas as pd

# Our training data (format: question → answer)
train_data = [
    {"instruction": "What is RAG?", "output": "RAG stands for Retrieval Augmented Generation. It combines a document retrieval system with an LLM. When a user asks a question, the retriever finds relevant documents from a knowledge base, and the LLM generates an answer based on those documents. This reduces hallucinations and allows access to up-to-date information."},
    {"instruction": "What is LoRA?", "output": "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique. Instead of updating all model weights, it adds small trainable rank decomposition matrices to attention layers. This reduces trainable parameters by about 99% while maintaining performance. LoRA makes fine-tuning accessible on consumer GPUs."},
    {"instruction": "How do transformers work?", "output": "The Transformer architecture processes all tokens in parallel using self-attention mechanisms. Each token generates Query, Key, and Value vectors. Attention scores are computed as softmax(Q*K^T/sqrt(d)) and used to weight the Values. Multiple attention heads learn different relationship types. Positional encoding preserves token order information."},
    {"instruction": "What is a vector database?", "output": "A vector database stores embeddings (numerical vector representations of data) and enables efficient similarity search. It uses ANN (Approximate Nearest Neighbor) algorithms like HNSW to find similar vectors quickly. Vector databases are essential for production RAG systems."},
    {"instruction": "Explain chain-of-thought prompting", "output": "Chain-of-thought (CoT) prompting encourages the model to reason step by step before giving a final answer. By adding 'Let's think step by step' to the prompt, the model breaks down complex problems into intermediate steps. This significantly improves performance on reasoning and math tasks."},
]

# Format in Alpaca-style instruction format
def format_alpaca(example):
    return {
        "text": f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    }

dataset = Dataset.from_pandas(pd.DataFrame(train_data))
dataset = dataset.map(format_alpaca)

print(f"Dataset: {len(dataset)} examples")
print("\nExample:")
print(dataset[0]["text"][:200] + "...")

In [ ]:
# @title 5. Train! (This takes ~10-20 min on T4)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,  # Small batch for T4
        gradient_accumulation_steps=4,  # Effective batch = 8
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=42,
    ),
)

trainer.train()
print("\nTraining complete!")

In [ ]:
# @title 6. Test the Fine-Tuned Model
FastLanguageModel.for_inference(model)

def generate_response(prompt):
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.1)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompt = """Below is an instruction that describes a task. Write a response.

### Instruction:
What is the difference between RAG and fine-tuning?

### Response:
"""

print(generate_response(test_prompt))

In [ ]:
# @title 7. Save the LoRA Adapter
# Save adapter (small file, ~16MB)
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")
print("LoRA adapter saved to 'lora_adapter/'")

# Download via Colab file browser
from google.colab import files
import shutil
shutil.make_archive("lora_adapter", 'zip', "lora_adapter")
# Uncomment to download:
# files.download("lora_adapter.zip")

## What's Next?

1. **Merge adapter** with base model for deployment
2. **Convert to GGUF** format for Ollama
3. Run your fine-tuned model locally!

→ Continue to `03-ollama-modelfile.md`